# Projet: Analyse des données et pre-processing

Eduardo Cobos Fernandez, Sipan Bareyan Jeremy Rakotorinira, Amine Berrahma 

Analyse et réduction des bias en Machine Learning

Dans ce notebook on va ...

# Introduction (explication du cas d’usage et de l’objectif)

**intro mi-projet edu :**

Les modèles de prédiction sont de plus en plus utilisés dans le domaine médical, notamment pour l’aide au diagnostic à partir de données patients et d’imagerie médicale. Toutefois, ces modèles peuvent hériter de biais présents dans les données utilisées pour leur apprentissage, ce qui rend nécessaire une analyse préalable de ces données.

Ce mi-projet s’appuie sur le dataset Chest X-ray NIH14 (https://www.kaggle.com/datasets/nih-chest-xrays/data), qui contient des radiographies thoraciques ainsi que des métadonnées associées aux patients, telles que l’âge, le genre et les pathologies diagnostiquées. Pour cette étape intermédiaire du projet, l’analyse se limite aux métadonnées, les images n’étant pas considérées. Un sous-ensemble de 15.000 radiographies est utilisé.

L’objectif de ce travail est d’analyser les métadonnées afin d’identifier d’éventuels biais liés à des attributs sensibles, puis d’appliquer une méthode de pré-processing pour réduire ces biais et construire un dataset transformé. Ce travail constitue une étape préparatoire au projet final, dans lequel l’impact du pré-processing sur un modèle basé sur les images sera étudié.


**intro mi-projet amine:**

L’intelligence artificielle joue un rôle important dans le domaine médical, notamment dans l’analyse d’images radiologiques. Les modèles de deep learning appliqués aux radiographies thoraciques peuvent assister les médecins dans le diagnostic de diverses maladies. Toutefois, ces systèmes peuvent reproduire ou amplifier des biais présents dans les données, ce qui peut conduire à des performances inéquitables selon certains groupes de patients.

Le cas d’usage étudié dans ce projet repose sur un sous-ensemble du dataset NIH Chest X-ray, composé de métadonnées associées à des radiographies thoraciques. Ce dataset contient des informations diverses (âge, genre...etc) ainsi que des annotations médicales indiquant la présence ou l’absence de maladies.

L’objectif principal de ce projet est d’identifier et d’analyser les biais potentiels présents dans ces données, en particulier en lien avec des attributs sensibles tels que le genre et l’âge. Pour cela, dans un premier temps une analyse descriptive est réalisée afin d’examiner la distribution des maladies selon ces groupes.

Dans un second temps, des métriques d’équité telles que la Statistical Parity Difference (SPD) et le Disparate Impact (DI) sont utilisées pour quantifier les déséquilibres observés.
Enfin, une méthode de mitigation par preprocessing, basée sur l’algorithme Reweighing, est appliquée afin de réduire ces biais et produire un dataset transformé plus équitable.

# Chargement des données

Les métadonnées du dataset Chest X-ray NIH14 sont chargées à partir du fichier fourni pour le projet. Ce fichier contient les informations associées à un sous-ensemble de 15 000 patients. (à modifier??)

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
from aif360.sklearn.metrics import *

import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from train_classifieur import train_classifier, pred_classifier

/Users/eduardo/Desktop/Cours/L3/Fairness/Mi-Projet/.venv/lib/python3.13/site-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/Users/eduardo/Desktop/Cours/L3/Fairness/Mi-Projet/.venv/lib/python3.13/site-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
# On commence par charger les donnees depuis le csv
df = pd.read_csv("Cobos_Fernandez_Eduardo.csv")      #j'ai chargé mes données, il faudra decider lequels utiliser

# On peut regarder les premieres lignes du dataset 
df.head()

NameError: name 'pd' is not defined

# 1 - Préparation des données :

Dans cette partie, une analyse est réalisée afin d’observer les dimensions du dataset, les types de variables ainsi que leur distribution. Une attention particulière est portée à la détection de valeurs abhérrantes, notamment sur la variable âge, afin de garantir la cohérence des informations.

Enfin, certaines transformations sont effectuées pour faciliter l’analyse, notamment la création d’une variable binaire indiquant la présence ou l’absence de maladie, ainsi que la catégorisation de l’âge en groupes. Ces étapes permettent d’obtenir un dataset propre, structuré et adapté à l’analyse des biais

# Informations sur le Dataset

In [ ]:
df.shape
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54252 entries, 0 to 54251
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Image Index                  54252 non-null  object 
 1   Finding Labels               54252 non-null  object 
 2   Follow-up #                  54252 non-null  int64  
 3   Patient ID                   54252 non-null  int64  
 4   Patient Age                  54252 non-null  int64  
 5   Patient Gender               54252 non-null  object 
 6   View Position                54252 non-null  object 
 7   OriginalImage[Width          54252 non-null  int64  
 8   Height]                      54252 non-null  int64  
 9   OriginalImagePixelSpacing[x  54252 non-null  float64
 10  y]                           54252 non-null  float64
dtypes: float64(2), int64(5), object(4)
memory usage: 4.6+ MB


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171000,0.171000
1,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143000,0.143000
2,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168000,0.168000
3,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168000,0.168000
4,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143000,0.143000
...,...,...,...,...,...,...,...,...,...,...,...
54247,00030793_000.png,Mass|Nodule,0,30793,58,F,PA,2021,2021,0.194311,0.194311
54248,00030794_000.png,No Finding,0,30794,38,F,PA,2021,2021,0.194311,0.194311
54249,00030796_000.png,No Finding,0,30796,44,M,PA,2021,2021,0.194311,0.194311
54250,00030798_000.png,No Finding,0,30798,30,M,PA,2500,2048,0.171000,0.171000


In [ ]:
df.isna().sum()

,0
Image Index,0
Finding Labels,0
Follow-up #,0
Patient ID,0
Patient Age,0
Patient Gender,0
View Position,0
OriginalImage[Width,0
Height],0
OriginalImagePixelSpacing[x,0


Ici on voit qu'il n'ya aucun Nan toutes nos cases ont des nombres, il n'y a pas de cellules vides ou manquantes. On confirme cela grace a la cellule df.isna.sum qui vaut 0 partout.

In [ ]:
df.describe()

,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
count,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000,54252.000000
mean,8.377829,14368.131663,46.702702,2648.508516,2488.283455,0.155542,0.155542
std,14.537982,8406.053815,17.071171,341.743128,400.517916,0.016171,0.016171
min,0.000000,2.000000,1.000000,1215.000000,966.000000,0.115000,0.115000
25%,0.000000,7262.750000,34.000000,2500.000000,2048.000000,0.143000,0.143000
50%,3.000000,14120.500000,49.000000,2530.000000,2544.000000,0.143000,0.143000
75%,9.000000,20680.000000,59.000000,2992.000000,2991.000000,0.168000,0.168000
max,156.000000,30802.000000,413.000000,3305.000000,3056.000000,0.198800,0.198800


In [ ]:
df.sort_values("Patient Age", ascending=False).head(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
41802,00021275_003.png,No Finding,3,21275,413,F,AP,3056,2544,0.139,0.139
41318,00021047_002.png,Mass|Pleural_Thickening,2,21047,412,M,AP,3056,2544,0.139,0.139
10219,00005567_000.png,Effusion|Pneumonia,0,5567,412,M,AP,3056,2544,0.139,0.139
41008,00020900_002.png,No Finding,2,20900,411,M,AP,3056,2544,0.139,0.139
47820,00026028_001.png,Atelectasis,1,26028,154,M,PA,2992,2991,0.143,0.143
46498,00025206_000.png,Infiltration|Mass,0,25206,153,M,PA,2992,2991,0.143,0.143
36029,00018366_044.png,Pneumothorax,44,18366,152,F,PA,2302,2991,0.143,0.143
30018,00015558_000.png,No Finding,0,15558,149,M,PA,2992,2991,0.143,0.143
23189,00012238_010.png,No Finding,10,12238,148,M,PA,2992,2991,0.143,0.143
27807,00014456_000.png,No Finding,0,14456,95,F,PA,2522,2991,0.143,0.143


In [ ]:
len(df["Finding Labels"].unique())

632

Grace aux 3 cellules précédentes : on voit que l'age maximal est de 413 ce qui n'est pas cohérent, c'est une valeur abhérrante. De plus on a 632 labels différents ce qui est enorme donc l'idée sera de transformer cette colonne en ou un label binaire:finding\no finding.

### Gestion des données abhérrantes

L’analyse descriptive de la variable Patient Age a mis en évidence la présence de valeurs abhérrantes, notamment des âges supérieurs à 120 ans, biologiquement improbables.

Plutôt que de supprimer directement les lignes dans le dataframe, une vérification a été effectuée au niveau de chaque Patient ID afin d’examiner si d’autres enregistrements du même patient comportaient un âge cohérent.

Lorsque c’était le cas, l’âge abhérrant a été corrigé en utilisant l’âge de référence du patient, défini comme l'âge le plus fréquent ou sinon la médiane des âges observés.

In [ ]:
df.duplicated().sum() #On verifie qu'il ya pas de doublons
df["Patient Gender"].unique() # On verifie que pour le genre il y'a M et F

array(['M', 'F'], dtype=object)

In [ ]:
df["Patient Age BEFORE"] = df["Patient Age"]# je garde une trace pour comparer plus tard

In [ ]:
def ref_age_func(s):
    s = s[(s >= 1) & (s <= 120)]
    m = s.mode()
    return m.iloc[0] if len(m) else s.median()

ref_age = df.groupby("Patient ID")["Patient Age"].apply(ref_age_func)

df.loc[df["Patient Age"] > 120, "Patient Age"] = df["Patient ID"].map(ref_age)

Visualisation du résultat


In [ ]:
df.loc[df["Patient Age BEFORE"] > 120, ["Patient ID", "Patient Age BEFORE", "Patient Age"]].head(20)



,Patient ID,Patient Age BEFORE,Patient Age
10219,5567,412,53
23189,12238,148,63
30018,15558,149,46
36029,18366,152,64
41008,20900,411,71
41318,21047,412,52
41802,21275,413,19
46498,25206,153,36
47820,26028,154,60


On voit bien ici que les ages ont été corrigés passant par exemple de 412 a 53 ans...